# RNA-seq Parquet Conversion

Streams each selected patient's raw RNA-seq count file from S3, filters to protein-coding genes, and writes one aggregated parquet file per group (Control / Tumeric) for downstream DESeq2 and modeling.

## Imports & S3 / DuckDB setup

In [ ]:
import pandas as pd, os,  sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from dotenv import load_dotenv
from src.s3_utils import RNA_to_parquet
load_dotenv("project.env")

True

## Convert RNA-seq files to parquet

In [4]:
selected_case_id = pd.read_csv(os.path.join(os.getenv("Output_path"), 'metadata.csv'))['file_id'].to_list()

RNA_to_parquet(
    BUCKET=os.getenv("AWS_BUCKET_NAME"), 
    SUBFOLDER_PATH="raw/RNA_seq/Control_data", 
    selected_case_id=selected_case_id
)

RNA_to_parquet(
    BUCKET=os.getenv("AWS_BUCKET_NAME"), 
    SUBFOLDER_PATH="raw/RNA_seq/Tumeric_data", 
    selected_case_id=selected_case_id
)

114 objects has been found under path: s3://clean-cancer-survival-bk/raw/RNA_seq/Control_data


Loading RNA-seq files:   0%|          | 0/114 [00:00<?, ?it/s]

Skipping non-expression file: raw/RNA_seq/Control_data/


Loading RNA-seq files: 100%|██████████| 114/114 [00:47<00:00,  2.40it/s]


1119 objects has been found under path: s3://clean-cancer-survival-bk/raw/RNA_seq/Tumeric_data


Loading RNA-seq files:   0%|          | 0/1119 [00:00<?, ?it/s]

Skipping non-expression file: raw/RNA_seq/Tumeric_data/


Loading RNA-seq files: 100%|██████████| 1119/1119 [04:17<00:00,  4.34it/s]


## Validate the parquet outputs

### Control

In [28]:
Control = pd.read_parquet(os.path.join(os.getenv("Output_path"), 'RNA_seq_expression_Control_data.parquet'), engine='fastparquet')

print(f"number of uniques values: {Control["file_id"].nunique()}")
print(f"shape of the parquet files: {Control.shape}")
print(f"number protein-coding gene counts {Control["gene_id"].nunique()}")  

number of uniques values: 64
shape of the parquet files: (1277568, 4)
number protein-coding gene counts 19962


### Tumeric

In [3]:
Tumeric = pd.read_parquet(os.path.join(os.getenv("Output_path"), 'RNA_seq_expression_Tumeric_data.parquet'), engine='fastparquet')

print(f"number of uniques values: {Tumeric["file_id"].nunique()}")
print(f"shape of the parquet files: {Tumeric.shape}")
print(f"number protein-coding gene counts {Tumeric["gene_id"].nunique()}")    

number of uniques values: 325
shape of the parquet files: (6487650, 4)
number protein-coding gene counts 19962


Final cohorts: 
- Control: 64 patients, including 19962 genes
- Tumeric: 325 patients, including 19962 genes

Define the ratio of tumeric and control patients for Power BI: 

In [39]:
ratio = pd.DataFrame({
    "Sample_type": ["Tumeric", "Control"],
    "count": [Tumeric["file_id"].nunique(), Control["file_id"].nunique()]
})
ratio["ratio"] = ratio["count"] / ratio["count"].sum()

ratio.to_csv(os.path.join(os.getenv("Output_path"), "ratio_Control_Tumeric.csv"))